In [2]:
import time
import difflib
import pandas as pd
from tqdm import tqdm

In [14]:
sales = pd.read_csv("data/raw/video_games_sales.csv")
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16717 non-null  object 
 1   Platform         16719 non-null  object 
 2   Year_of_Release  16450 non-null  float64
 3   Genre            16717 non-null  object 
 4   Publisher        16665 non-null  object 
 5   NA_Sales         16719 non-null  float64
 6   EU_Sales         16719 non-null  float64
 7   JP_Sales         16719 non-null  float64
 8   Other_Sales      16719 non-null  float64
 9   Global_Sales     16719 non-null  float64
 10  Critic_Score     8137 non-null   float64
 11  Critic_Count     8137 non-null   float64
 12  User_Score       10015 non-null  object 
 13  User_Count       7590 non-null   float64
 14  Developer        10096 non-null  object 
 15  Rating           9950 non-null   object 
dtypes: float64(9), object(7)
memory usage: 2.0+ MB


## Fixing dev name

In [3]:
indie_devs = pd.read_csv("data/raw/indie_games_developers.csv")
other_devs = pd.read_csv("data/raw/video_games_developers.csv")
devs = pd.concat([indie_devs, other_devs[indie_devs.columns]]).drop_duplicates(subset='Developer')
devs.to_csv("data/processed/unified_videogames_devs_and_publishers.csv")
devs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 830 entries, 0 to 685
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Developer        830 non-null    object
 1   City             813 non-null    object
 2   Autonomous area  412 non-null    object
 3   Country          830 non-null    object
 4   Notable games    751 non-null    object
 5   Notes            444 non-null    object
dtypes: object(6)
memory usage: 45.4+ KB


In [4]:
unmatched_developers = sales[~sales["Developer"].isin(devs["Developer"])]["Developer"].unique()
print(len(unmatched_developers))

1379


In [ ]:
no_match = []
skipped = []
dev_list = devs["Developer"].values
for unmatched_dev in tqdm(unmatched_developers):
    if pd.notna(unmatched_dev) and unmatched_dev not in skipped:
        time.sleep(0.25)
        match = difflib.get_close_matches(unmatched_dev, dev_list, n=1, cutoff=0.6)
        try:
            if match[0] and input(unmatched_dev + " -> " + match[0]) == "y":
                sales.loc[sales["Developer"] == unmatched_dev, "Developer"] = match[0]
            else:
                skipped.append(unmatched_dev)
        except IndexError as e:
            no_match.append(unmatched_dev)

In [ ]:
for skipped_dev in tqdm(skipped):
    time.sleep(0.25)
    match = difflib.get_close_matches(skipped_dev, dev_list, n=1, cutoff=0.75)
    try:
        if match[0] and input(skipped_dev + " -> " + match[0]) == "y":
            sales.loc[sales["Developer"] == skipped_dev, "Developer"] = match[0]
    except IndexError as e:
            no_match.append(unmatched_dev)

In [ ]:
sales.to_csv("./data/processed/processed_video_game_sales.csv")

## Adding the location of developer/publisher to sales data

### Filling null devs with publisher

In [19]:
processed_devs = pd.read_csv("data/processed/unified_videogames_devs_and_publishers.csv")
processed_sales = pd.read_csv("data/processed/processed_video_game_sales.csv")
processed_sales['Developer'] = processed_sales['Developer'].fillna(processed_sales['Publisher'])
processed_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       16719 non-null  int64  
 1   Name             16717 non-null  object 
 2   Platform         16719 non-null  object 
 3   Year_of_Release  16450 non-null  float64
 4   Genre            16717 non-null  object 
 5   Publisher        16665 non-null  object 
 6   NA_Sales         16719 non-null  float64
 7   EU_Sales         16719 non-null  float64
 8   JP_Sales         16719 non-null  float64
 9   Other_Sales      16719 non-null  float64
 10  Global_Sales     16719 non-null  float64
 11  Critic_Score     8137 non-null   float64
 12  Critic_Count     8137 non-null   float64
 13  User_Score       10015 non-null  object 
 14  User_Count       7590 non-null   float64
 15  Developer        16674 non-null  object 
 16  Rating           9950 non-null   object 
dtypes: float64(9

In [ ]:
result = processed_sales.merge(
    processed_devs[['Developer', 'Country']],
    on='Developer',
    how='left',
    validate='m:1'
)

result = result.rename(columns={'Country': 'Developer/Publisher country'})
result.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   16719 non-null  int64  
 1   Name                         16717 non-null  object 
 2   Platform                     16719 non-null  object 
 3   Year_of_Release              16450 non-null  float64
 4   Genre                        16717 non-null  object 
 5   Publisher                    16665 non-null  object 
 6   NA_Sales                     16719 non-null  float64
 7   EU_Sales                     16719 non-null  float64
 8   JP_Sales                     16719 non-null  float64
 9   Other_Sales                  16719 non-null  float64
 10  Global_Sales                 16719 non-null  float64
 11  Critic_Score                 8137 non-null   float64
 12  Critic_Count                 8137 non-null   float64
 13  User_Score      

### Swapping devs with null location for the publisher

In [ ]:
where_location_is_null = result['Developer/Publisher country'].isna()
result.loc[where_location_is_null, "Developer"] = result.loc[where_location_is_null, 'Publisher']
result = result.drop(columns=["Developer/Publisher country"])
result = result.merge(
    processed_devs[['Developer', 'Country']],
    on='Developer',
    how='left'
)
result = result.rename(columns={'Country': 'Developer/Publisher country'})
result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   16719 non-null  int64  
 1   Name                         16717 non-null  object 
 2   Platform                     16719 non-null  object 
 3   Year_of_Release              16450 non-null  float64
 4   Genre                        16717 non-null  object 
 5   Publisher                    16665 non-null  object 
 6   NA_Sales                     16719 non-null  float64
 7   EU_Sales                     16719 non-null  float64
 8   JP_Sales                     16719 non-null  float64
 9   Other_Sales                  16719 non-null  float64
 10  Global_Sales                 16719 non-null  float64
 11  Critic_Score                 8137 non-null   float64
 12  Critic_Count                 8137 non-null   float64
 13  User_Score      

### Dropping rows with null location

In [ ]:
final_dataset = result.dropna(subset='Developer/Publisher country')
final_dataset.to_csv("data/final/final_videogame_sales.csv")
final_dataset.info()

## Treating Country Names

In [ ]:
final_dataset = pd.read_csv("data/final/final_videogame_sales.csv").drop(columns=["Unnamed: 0", "Unnamed: 0.1"])
print(final_dataset["Developer/Publisher country"].unique())

['Japan' 'United States' 'Europe' 'Canada' 'United Kingdom' 'Poland'
 'Sweden' 'Australia' 'France' 'Netherlands' 'Denmark'
 'United Kingdom (England)' 'Czech Republic' 'Romania' 'Germany' 'Finland'
 'Singapore' 'Spain' 'Russia' 'UkraineMalta' 'Hungary' 'Italy'
 'South Korea' 'Bulgaria' 'CyprusBelarus' 'Slovenia' 'Ukraine' 'Belgium'
 'Austria' 'Slovakia' 'Norway' 'Croatia' 'Taiwan']


In [ ]:
# Todos os jogos cujo país de origem é "Europe" são da Ubisoft, então o local será trocado pela frança
final_dataset.loc[final_dataset["Developer/Publisher country"] == "Europe", "Developer"].unique()

array(['Ubisoft'], dtype=object)

In [26]:
correction_map = {
    "United States": "United States of America",
    "Russia": "Russian Federation",
    "Czech Republic": "Czechia",
    
    "United Kingdom (England)": "United Kingdom",
    
    "UkraineMalta": "Ukraine",
    "CyprusBelarus": "Belarus",
    
    # Ubisoft
    "Europe": "France"
}

final_dataset["Developer/Publisher country"] = final_dataset["Developer/Publisher country"].replace(correction_map)
print(final_dataset["Developer/Publisher country"].unique())

['Japan' 'United States of America' 'France' 'Canada' 'United Kingdom'
 'Poland' 'Sweden' 'Australia' 'Netherlands' 'Denmark' 'Czechia' 'Romania'
 'Germany' 'Finland' 'Singapore' 'Spain' 'Russian Federation' 'Ukraine'
 'Hungary' 'Italy' 'South Korea' 'Bulgaria' 'Belarus' 'Slovenia' 'Belgium'
 'Austria' 'Slovakia' 'Norway' 'Croatia' 'Taiwan']


In [27]:
final_dataset.to_csv("data/final/final_videogame_sales.csv")

## Treating Genres

In [30]:
final_dataset = pd.read_csv("data/final/final_videogame_sales.csv")
final_dataset["Genre"].unique()

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', nan], dtype=object)

In [31]:
final_dataset.loc[final_dataset['Genre'].isna(), "Genre"] = "Other"
final_dataset.loc[final_dataset['Genre'] == "Misc", "Genre"] = "Other"
final_dataset["Genre"].unique()

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Other',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy'], dtype=object)

In [32]:
final_dataset.to_csv("data/final/final_videogame_sales.csv")

## Generating secondary dataset: Sales by developer country

In [32]:
final_dataset = pd.read_csv("data/final/final_videogame_sales.csv")

sales_by_dev_country = final_dataset.groupby('Developer/Publisher country')['Global_Sales'].sum().reset_index()
sales_by_dev_country.columns = ['Country', 'Copies Sold']
sales_by_dev_country = sales_by_dev_country.sort_values(by='Copies Sold', ascending=False)

sales_by_dev_country.to_csv("data/final/sales_by_dev_country.csv")

## Adding backloggd user ratings to games with matching name

In [92]:
final_dataset = pd.read_csv("data/final/final_videogame_sales.csv")
final_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11148 entries, 0 to 11147
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   11148 non-null  int64  
 1   Unnamed: 0.2                 11148 non-null  int64  
 2   Name                         11146 non-null  object 
 3   Platform                     11148 non-null  object 
 4   Year_of_Release              10992 non-null  float64
 5   Genre                        11148 non-null  object 
 6   Publisher                    11144 non-null  object 
 7   NA_Sales                     11148 non-null  float64
 8   EU_Sales                     11148 non-null  float64
 9   JP_Sales                     11148 non-null  float64
 10  Other_Sales                  11148 non-null  float64
 11  Global_Sales                 11148 non-null  float64
 12  Critic_Score                 6782 non-null   float64
 13  Critic_Count    

In [93]:
backloggd_games = pd.read_csv("data/raw/backloggd_games.csv")
backloggd_games["rating"] = backloggd_games["rating"] * 2
backloggd_games.head()

,id,name,date,rating,reviews,plays,playing,backlogs,wishlists,description
0,1000001,Cathode Ray Tube Amusement Device,1947-12-31,7.2,85,149,1,42,72,The cathode ray tube amusement device is the e...
1,1000002,Bertie the Brain,1950-08-25,6.0,26,46,0,9,17,Currently considered the first videogame in hi...
2,1000003,Nim,1951-12-31,3.8,9,26,0,2,8,The Nimrod was a special purpose computer that...
3,1000004,Draughts,1952-08-31,5.6,9,30,0,4,7,A game of draughts (a.k.a. checkers) written f...
4,1000005,OXO,1952-12-31,6.2,22,80,0,11,15,OXO was a computer game developed by Alexander...


In [94]:
filtro = final_dataset["Name"].isin(backloggd_games["name"])
final_dataset[filtro.__and__(final_dataset["User_Score"].isna())].info()

<class 'pandas.core.frame.DataFrame'>
Index: 1716 entries, 1 to 11146
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   1716 non-null   int64  
 1   Unnamed: 0.2                 1716 non-null   int64  
 2   Name                         1714 non-null   object 
 3   Platform                     1716 non-null   object 
 4   Year_of_Release              1687 non-null   float64
 5   Genre                        1716 non-null   object 
 6   Publisher                    1715 non-null   object 
 7   NA_Sales                     1716 non-null   float64
 8   EU_Sales                     1716 non-null   float64
 9   JP_Sales                     1716 non-null   float64
 10  Other_Sales                  1716 non-null   float64
 11  Global_Sales                 1716 non-null   float64
 12  Critic_Score                 20 non-null     float64
 13  Critic_Count          

In [95]:
mapa_ratings = backloggd_games.set_index("name")["rating"]
mapa_ratings = mapa_ratings[~mapa_ratings.index.duplicated(keep='first')]
filtro_atualizar = (final_dataset["User_Score"].isna()) & (final_dataset["Name"].isin(backloggd_games["name"]))
final_dataset.loc[filtro_atualizar, "User_Score"] = final_dataset.loc[filtro_atualizar, "Name"].map(mapa_ratings)

In [96]:
final_dataset.head()

,Unnamed: 0,Unnamed: 0.2,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating,Developer/Publisher country
0,0,0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E,Japan
1,1,1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,7.0,NaN,Nintendo,NaN,Japan
2,2,2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E,Japan
3,3,3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E,Japan
4,4,4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,7,NaN,Nintendo,NaN,Japan


## Removing games with no user rating that did not sell well (less than 0.5M)

In [100]:
filtro_atualizar = (final_dataset["User_Score"].isna()) & (final_dataset["Global_Sales"] < 0.5)
final_dataset = final_dataset[~filtro_atualizar].reset_index()
final_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9817 entries, 0 to 9816
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   level_0                      9817 non-null   int64  
 1   index                        9817 non-null   int64  
 2   Unnamed: 0                   9817 non-null   int64  
 3   Unnamed: 0.2                 9817 non-null   int64  
 4   Name                         9816 non-null   object 
 5   Platform                     9817 non-null   object 
 6   Year_of_Release              9675 non-null   float64
 7   Genre                        9817 non-null   object 
 8   Publisher                    9814 non-null   object 
 9   NA_Sales                     9817 non-null   float64
 10  EU_Sales                     9817 non-null   float64
 11  JP_Sales                     9817 non-null   float64
 12  Other_Sales                  9817 non-null   float64
 13  Global_Sales      

In [101]:
final_dataset.to_csv("data/final/final_videogame_sales.csv")